# Products - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_products"
target_table = f"{catalog}.silver.olist_products"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
translation_table = (f"{catalog}.silver.olist_product_category_translation")

translation_df = spark.table(translation_table)

silver_df = (
    bronze_df.alias("p")
    .join(
        translation_df.alias("t"),
        col("p.product_category_name")
        == col("t.product_category_name_portuguese"),
        "left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name").alias("product_category_name_portuguese"),
        col("t.product_category_name_english"),
        col("p.product_name_lenght").alias("product_name_length"),
        col("p.product_description_lenght").alias("product_description_length"),
        col("p.product_photos_qty").alias("product_photo_count"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm"),
        col("p._rescued_data"),
        col("p.source_file_path"),
        col("p.source_file_modification_time"),
        col("p.ingestion_timestamp"),
        col("p.ingestion_run_id"),
        col("p.source_system"),
        col("p.source_dataset")
    )
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)